In [5]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import re
import pandas as pd
import json
import numpy as np
import faiss

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
TRANSCRIPTS_DIR = DATA_DIR / "Transcripts"
QA_DIR = DATA_DIR / "QA"

transcript_files = sorted(TRANSCRIPTS_DIR.glob("*.txt"))
qa_files = sorted(QA_DIR.glob("*.json"))

print("Transcripts folder exists:", TRANSCRIPTS_DIR.exists())
print("QA folder exists:", QA_DIR.exists())
print("Number of transcript files:", len(transcript_files))
print("Number of QA files:", len(qa_files))

Transcripts folder exists: True
QA folder exists: True
Number of transcript files: 13
Number of QA files: 7


In [6]:
def load_transcripts(transcript_files):
    rows = []

    for file_path in transcript_files:
        episode_name = file_path.stem

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                match = re.match(r"^([\d.]+):\s*(.*)$", line)

                if match:
                    timestamp = match.group(1)
                    text = match.group(2).strip()

                    if text:
                        rows.append({
                            "episode": episode_name,
                            "timestamp": timestamp,
                            "text": text
                        })

    return pd.DataFrame(rows)


transcripts_df = load_transcripts(transcript_files)

print("Total transcript rows:", len(transcripts_df))
display(transcripts_df.head())

display(
    transcripts_df.groupby("episode")
    .size()
    .reset_index(name="num_lines")
)

Total transcript rows: 11109


,episode,timestamp,text
0,أعظم طائرة حربية الدحيح,0.0,سيادة الكولونيل، صبرك في محله،
1,أعظم طائرة حربية الدحيح,3.076,مبروك علينا،
2,أعظم طائرة حربية الدحيح,4.238,"عملنا أفجر طيارة في تاريخ ""أمريكا""."
3,أعظم طائرة حربية الدحيح,6.184,أنا متحمس جدًا من امبارح،
4,أعظم طائرة حربية الدحيح,8.308,ها، ورّيني!


,episode,num_lines
0,أعظم طائرة حربية الدحيح,761
1,الأخطبوط الدحيح,784
2,الساموراي الدحيح,662
3,تاج محل الدحيح,547
4,جون كينيدي الدحيح,1125
5,فيزياء و فلسفة الحركة الدحيح,955
6,كيف تحولت روسيا إلى إمبراطورية؟ الدحيح,1166
7,كيف تسيطر على عقول البشر؟ الدحيح,908
8,كيف تنقل جبل وزنه 30 طن قبل أن يغرق؟ الدحيح,757
9,مصير الأرض و الشمس و كل شيء الدحيح,656


In [7]:
AR_PUNCT_MAP = {
    ",": "،",
    ";": "؛",
    "?": "؟",
    "\"": "«",
    "“": "«",
    "”": "»"
}

PUNCT_RE = re.compile("|".join(re.escape(k) for k in AR_PUNCT_MAP.keys()))

def strip_timestamp(line: str) -> str:
    timestamp_pattern = r"^(\d+[\.:]\d+([\.:]\d+)?[:]?)\s*"
    return re.sub(timestamp_pattern, "", line).strip()

def remove_noise_tags(text: str) -> str:
    text = re.sub(r"\[.*?\]", "", text)
    return text.strip()

def normalize_ms3_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = strip_timestamp(text)
    text = remove_noise_tags(text)

    # Standardize common punctuation, but do NOT remove punctuation
    text = PUNCT_RE.sub(lambda m: AR_PUNCT_MAP[m.group(0)], text)

    # Keep English tokens, only separate Arabic-English boundaries
    text = re.sub(r"([\u0600-\u06FF])([A-Za-z\d])", r"\1 \2", text)
    text = re.sub(r"([A-Za-z\d])([\u0600-\u06FF])", r"\1 \2", text)

    # Clean spaces only
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [9]:
transcripts_df["normalized_text"] = transcripts_df["text"].apply(normalize_ms3_text)

print("normalized_text column created:", "normalized_text" in transcripts_df.columns)

display(transcripts_df[["episode", "timestamp", "text", "normalized_text"]].head())

print("Original sample:")
print(transcripts_df["text"].iloc[0])

print("\nNormalized sample:")
print(transcripts_df["normalized_text"].iloc[0])

normalized_text column created: True


,episode,timestamp,text,normalized_text
0,أعظم طائرة حربية الدحيح,0.0,سيادة الكولونيل، صبرك في محله،,سيادة الكولونيل، صبرك في محله،
1,أعظم طائرة حربية الدحيح,3.076,مبروك علينا،,مبروك علينا،
2,أعظم طائرة حربية الدحيح,4.238,"عملنا أفجر طيارة في تاريخ ""أمريكا"".",عملنا أفجر طيارة في تاريخ «أمريكا«.
3,أعظم طائرة حربية الدحيح,6.184,أنا متحمس جدًا من امبارح،,أنا متحمس جدًا من امبارح،
4,أعظم طائرة حربية الدحيح,8.308,ها، ورّيني!,ها، ورّيني!


Original sample:
سيادة الكولونيل، صبرك في محله،

Normalized sample:
سيادة الكولونيل، صبرك في محله،


In [10]:
CHUNK_SIZE = 12
CHUNK_OVERLAP = 3
MIN_WORDS = 20

chunks = []

for episode in transcripts_df["episode"].unique():

    episode_df = transcripts_df[
        transcripts_df["episode"] == episode
    ].reset_index(drop=True)

    texts = episode_df["normalized_text"].tolist()
    timestamps = episode_df["timestamp"].tolist()

    start = 0

    while start < len(texts):

        end = min(start + CHUNK_SIZE, len(texts))

        chunk_text = " ".join(texts[start:end])

        chunk = {
            "episode": episode,
            "start_timestamp": timestamps[start],
            "end_timestamp": timestamps[end -1],
            "chunk_text": chunk_text
        }

        chunks.append(chunk)

        start += CHUNK_SIZE - CHUNK_OVERLAP

chunks_df = pd.DataFrame(chunks)

# Chunk statistics
chunks_df["num_words"] = chunks_df["chunk_text"].apply(
    lambda x: len(x.split())
)

chunks_df["num_chars"] = chunks_df["chunk_text"].apply(len)

# Remove very short chunks
chunks_df = chunks_df[
    chunks_df["num_words"] >= MIN_WORDS
].reset_index(drop=True)

print("Total chunks:", len(chunks_df))
print("Average words per chunk:", round(chunks_df["num_words"].mean(), 2))
print("Minimum words:", chunks_df["num_words"].min())
print("Maximum words:", chunks_df["num_words"].max())

display(chunks_df.head())

Total chunks: 1236
Average words per chunk: 63.37
Minimum words: 24
Maximum words: 94


,episode,start_timestamp,end_timestamp,chunk_text,num_words,num_chars
0,أعظم طائرة حربية الدحيح,0.0,25.494,سيادة الكولونيل، صبرك في محله، مبروك علينا، عم...,58,320
1,أعظم طائرة حربية الدحيح,22.165,47.616,يعني إيه سنين ضوئية؟! مش مهم، مش مهم، احكيلي ع...,62,339
2,أعظم طائرة حربية الدحيح,44.34,63.501,لو افترضنا إن هناك شخص، ومثلًا مثلًا مثلًا، يع...,57,319
3,أعظم طائرة حربية الدحيح,60.783,80.211,وصعب أي فرد يتتبعها على الـ... سؤال من واحد صا...,51,266
4,أعظم طائرة حربية الدحيح,77.405,97.433,ثانية واحدة! إيه 7 لغات دي؟! اوعى يكون فيها لغ...,55,287


In [11]:
sample_chunk = chunks_df.iloc[0]

print("Episode:")
print(sample_chunk["episode"])

print("\nTimestamps:")
print(sample_chunk["start_timestamp"], "->", sample_chunk["end_timestamp"])

print("\nChunk:")
print(sample_chunk["chunk_text"])

Episode:
أعظم طائرة حربية  الدحيح

Timestamps:
0.0 -> 25.494

Chunk:
سيادة الكولونيل، صبرك في محله، مبروك علينا، عملنا أفجر طيارة في تاريخ «أمريكا«. أنا متحمس جدًا من امبارح، ها، ورّيني! أقدم لحضرتك فخر الطيران الأمريكي الـ F-35. بس دي شكلها شبه اللي احنا عملناها قبل كدا! آه، بس التكنولوجيا اللي فيها يعني إيه سنين ضوئية؟! مش مهم، مش مهم، احكيلي عنها كدا. القطعة الفنية اللي أمام حضرتك دي


In [19]:
# ==========================================
# EMBEDDING MODEL
# ==========================================

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

# ==========================================
# GENERATE EMBEDDINGS
# ==========================================

texts = chunks_df["chunk_text"].tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)

# ==========================================
# BUILD FAISS VECTOR STORE
# ==========================================

embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

index.add(embeddings)

print("Vectors stored in FAISS:", index.ntotal)

# ==========================================
# RETRIEVAL FUNCTION
# ==========================================

def retrieve_chunks(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        row = chunks_df.iloc[idx]

        results.append({
            "score": float(score),
            "episode": row["episode"],
            "start_timestamp": row["start_timestamp"],
            "end_timestamp": row["end_timestamp"],
            "chunk_text": row["chunk_text"]
        })

    return results

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Embeddings shape: (1236, 384)
Vectors stored in FAISS: 1236


In [ ]:
query = "ما هي تقنية Deep Focus في فيلم Citizen Kane؟"

results = retrieve_chunks(query, top_k=3)

for i, result in enumerate(results, start=1):

    print(f"\nResult {i}")
    print("Score:", round(result["score"], 4))
    print("Episode:", result["episode"])
    print("Time:", result["start_timestamp"], "->", result["end_timestamp"])

    print("\nChunk:")
    print(result["chunk_text"][:500])


Result 1
Semantic score: 0.6582
Keyword score: 0.4444
Final score: 0.6262
Episode: هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح
Time: 974.201 -> 1005.006

Chunk:
عشان الرسّام يحدد ويركّز على التفاصيل، أو إن الشخصية المطلوبة تتبروز وتبان. هما استخدموا التقنية دي والإضاءة ما تبقاش شكل حلو بس، «لأ، احنا هنا بنستعملها فيه، يا عزيزي، سينما كاملة وعشان تخلّيك كمُشاهد ونُقّاد كتير بيعتبروا إن Citizen Kane هو الفيلم اللي مهّد الطريق وبعيدًا عن الـ Film Noir ، كل فيلم بيعتمد في كادراته على النور والضِل، عشان يحكيلك عن عُزلة البطل

Result 2
Semantic score: 0.67
Keyword score: 0.3333
Final score: 0.6195
Episode: هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح
Time: 1569.254 -> 1600.051

Chunk:
ولسة بيدفع تَمَن Citizen Kane نهاية، يا عزيزي، هذا الصراع سنة 2015 ، بيتعرض فيلم Citizen Kane في قصر «هيرست« نفسه، بموافقة «ستيفن هيرست«، اللي أكّد في النهاية إن دا فيلم عظيم، فيلم سينمائي مهم، ولكن في تاريخ السينما. ودي، يا عزيزي، كانت لحظة شاعرية، واعتراف إن الفن الجيد بيفرض نفسه في الآخر، حتى لو داخل قصر

In [20]:
def build_context(retrieved_chunks):

    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):

        source = (
            f"[Source {i}] "
            f"Episode: {chunk['episode']} | "
            f"Time: {chunk['start_timestamp']} - {chunk['end_timestamp']}"
        )

        text = chunk["chunk_text"]

        context_parts.append(
            source + "\n" + text
        )

    return "\n\n".join(context_parts)

In [21]:
query = "ما هي تقنية Deep Focus؟"

retrieved = retrieve_chunks(query, top_k=3)

context = build_context(retrieved)

print(context[:3000])

[Source 1] Episode: هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح | Time: 742.927 - 768.418
عشان كدا، الـ Deep Focus لم تعُد دي بقت فلسفة كاملة في الإخراج، العمق البصري بقى مرادف للعمق النفسي، الصورة بقت مراية للعقل، مش للعين بس. زي ما قُلتلك، يا عزيزي، لأنه حرّك الكاميرا، عشان تخدم الحدوتة، بدل ما كانت مجرد أداة ثابتة وبتصور. بس «ويلز« مش بس حركها، دا حوّل الكاميرا بيزحف، بيطير، بيتسلل زي الشبح، الكاميرا بقت بتتحرك حركة مستحيلة، كأنها طائر،

[Source 2] Episode: هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح | Time: 722.178 - 747.294
كل المعاني بتحصل قُدّامك في لَقطة واحدة. طبعًا، التأثير اللي «ويلز« و«تولاند« بقى «أوبشن« عادي جدًا بس الفَرْق إن «ويلز« كان بيخترع دا عشان يقدر يعمل تركيز وعمق بصري، يخلّي عينك انت هي اللي تمسح الكادر وتختار هتركّز على إيه، كأن «ويلز« كان بيخلق قبل ما الـ«سوشيال ميديا« نفسها عشان كدا، الـ Deep Focus لم تعُد دي بقت فلسفة كاملة في الإخراج، العمق البصري بقى مرادف للعمق النفسي،

[Source 3] Episode: فيزياء و فلسفة الحركة  الدحيح | Time: 1986.744 - 2012.959
دي أ

In [22]:
# ==========================================
# SYSTEM PROMPT
# ==========================================

SYSTEM_PROMPT = """
أنت مساعد ذكي يعتمد فقط على المعلومات الموجودة في السياق المسترجع.

قواعد مهمة:
- أجب فقط باستخدام المعلومات الموجودة في السياق.
- إذا لم تجد الإجابة في السياق، قل:
"لا أملك معلومات كافية للإجابة من البيانات المتاحة."
- لا تخترع معلومات غير موجودة.
- يمكنك الإجابة بالعربية أو الإنجليزية حسب لغة السؤال.
- حاول أن تكون الإجابة واضحة ومختصرة.
"""

# ==========================================
# CONTEXT CONSTRUCTION
# ==========================================

def build_context(retrieved_chunks):

    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):

        source = (
            f"[Source {i}] "
            f"Episode: {chunk['episode']} | "
            f"Time: {chunk['start_timestamp']} - "
            f"{chunk['end_timestamp']}"
        )

        text = chunk["chunk_text"]

        context_parts.append(
            source + "\n" + text
        )

    return "\n\n".join(context_parts)

# ==========================================
# FINAL PROMPT CONSTRUCTION
# ==========================================

def build_prompt(query, context):

    prompt = f"""
{SYSTEM_PROMPT}

السياق:
{context}

السؤال:
{query}

الإجابة:
"""

    return prompt

In [23]:
query = "ما هي تقنية Deep Focus؟"

retrieved = retrieve_chunks(query, top_k=3)

context = build_context(retrieved)

prompt = build_prompt(query, context)

print(prompt[:4000])



أنت مساعد ذكي يعتمد فقط على المعلومات الموجودة في السياق المسترجع.

قواعد مهمة:
- أجب فقط باستخدام المعلومات الموجودة في السياق.
- إذا لم تجد الإجابة في السياق، قل:
"لا أملك معلومات كافية للإجابة من البيانات المتاحة."
- لا تخترع معلومات غير موجودة.
- يمكنك الإجابة بالعربية أو الإنجليزية حسب لغة السؤال.
- حاول أن تكون الإجابة واضحة ومختصرة.


السياق:
[Source 1] Episode: هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح | Time: 742.927 - 768.418
عشان كدا، الـ Deep Focus لم تعُد دي بقت فلسفة كاملة في الإخراج، العمق البصري بقى مرادف للعمق النفسي، الصورة بقت مراية للعقل، مش للعين بس. زي ما قُلتلك، يا عزيزي، لأنه حرّك الكاميرا، عشان تخدم الحدوتة، بدل ما كانت مجرد أداة ثابتة وبتصور. بس «ويلز« مش بس حركها، دا حوّل الكاميرا بيزحف، بيطير، بيتسلل زي الشبح، الكاميرا بقت بتتحرك حركة مستحيلة، كأنها طائر،

[Source 2] Episode: هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح | Time: 722.178 - 747.294
كل المعاني بتحصل قُدّامك في لَقطة واحدة. طبعًا، التأثير اللي «ويلز« و«تولاند« بقى «أوبشن« عادي جدًا بس الفَ

In [24]:
from dotenv import load_dotenv
import os

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

print("Google API key loaded:", GOOGLE_API_KEY is not None)

Google API key loaded: False
